In [2]:
import os

# Find project root (go up two levels from routing/)

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

print ("Project root:", project_root)

#Build full path to geopackage

data_path = os.path.join (project_root, "1_dataset", "raw", "processed", "lagos_data.gpkg")

print ("GPKG path:", data_path)

Project root: /home/ridwan-ayinde/Desktop/MIT Emerging Talent/Final Project
GPKG path: /home/ridwan-ayinde/Desktop/MIT Emerging Talent/Final Project/1_dataset/raw/processed/lagos_data.gpkg


In [3]:
# --- BUILD ROAD NETWORK GRAPH ---

import geopandas as gpd
import fiona

print("Layers:", fiona.listlayers(data_path))

roads = gpd.read_file(data_path, layer='roads_clean')

roads.head()

Layers: ['lagos_boundaryshp__nga_admbnda_adm2_osgof_20170222', 'roads_lagos', 'buildings_lagos', 'pois_lagos', 'buildings_clean', 'roads_clean', 'pois_clean']


,geometry
0,"MULTILINESTRING ((3.4039 6.45847, 3.40274 6.45..."
1,"MULTILINESTRING ((3.40403 6.44011, 3.40472 6.4..."
2,"MULTILINESTRING ((3.38844 6.46564, 3.38792 6.4..."
3,"MULTILINESTRING ((3.39968 6.45921, 3.39912 6.4..."
4,"MULTILINESTRING ((3.408 6.45503, 3.40795 6.454..."


In [4]:
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, MultiLineString

print ('Loading roads from Lagos GeoPackage...')

# We load only the Roads layer from the same geopackage

roads_clean_final = gpd.read_file(data_path, layer="roads_clean")

print (roads_clean_final.head())
print (f"CRS: {roads_clean_final.crs}")

Loading roads from Lagos GeoPackage...
                                            geometry
0  MULTILINESTRING ((3.4039 6.45847, 3.40274 6.45...
1  MULTILINESTRING ((3.40403 6.44011, 3.40472 6.4...
2  MULTILINESTRING ((3.38844 6.46564, 3.38792 6.4...
3  MULTILINESTRING ((3.39968 6.45921, 3.39912 6.4...
4  MULTILINESTRING ((3.408 6.45503, 3.40795 6.454...
CRS: EPSG:4326


In [5]:
# Reproject to metric CRS (UTM for Lagos ~ EPSG:32632 or Web Mercator EPSG:3857)

roads_clean_final_m = roads_clean_final.to_crs(epsg=3857)

print("Reprojected CRS:", roads_clean_final_m.crs)

Reprojected CRS: EPSG:3857


In [6]:
def roads_to_edges(gdf):
    edges = []
    
    for idx, row in gdf.iterrows():
        geom = row.geometry
        
        if isinstance (geom, LineString):
            coords = list (geom.coords)
        elif isinstance (geom, MultiLineString):
            coords = []
            for line in geom.geoms:
                coords += list(line.coords)
        else:
            continue
        
        for i in range(len(coords)-1):
            u = coords[i]
            v = coords[i+1]
            length = LineString([u, v]).length
            edges.append((u,v, {"length":length}))
            
    return edges

edges = roads_to_edges (roads_clean_final_m)

G = nx.Graph()
G.add_edges_from(edges)

print ("Graph built successfully")
print (f"Number of nodes: {G.number_of_nodes()}")
print (f"Number of edges: {G.number_of_edges()}")

Graph built successfully
Number of nodes: 15248
Number of edges: 16310


In [7]:
# Load POIs Layer

pois_clean = gpd.read_file(data_path, layer='pois_clean')
print ("Loaded POIs:")
print (pois_clean.head())
print ("CRS:",pois_clean.crs)

Loaded POIs:
         fclass                                           geometry
0  market_place  MULTIPOLYGON (((3.38331 6.4626, 3.38336 6.4626...
1  market_place  MULTIPOLYGON (((3.3912 6.46051, 3.39146 6.4606...
2          park  MULTIPOLYGON (((3.39585 6.44864, 3.39638 6.449...
3   arts_centre  MULTIPOLYGON (((3.39428 6.44899, 3.39446 6.449...
4         pitch  MULTIPOLYGON (((3.39449 6.45098, 3.39466 6.451...
CRS: EPSG:4326


In [8]:
# Making sure that POIs are in the same CRS as the graph

pois_clean_m = pois_clean.to_crs (roads_clean_final_m.crs)

In [10]:
# Create a KDTree of graph nodes for fast nearest neighbor search

import numpy as np
from scipy.spatial import cKDTree

# Get list of all nodes coordinates

nodes = np.array(G.nodes)

# Build KDTree for fast nearest mode lookup
tree = cKDTree (nodes)

In [15]:
pois_points = pois_clean_m.copy()
pois_points ["geometry"] = pois_points.geometry.centroid

print (pois_points.head())
print (pois_points.geometry.iloc[0])

         fclass                       geometry
0  market_place  POINT (376779.993 720951.273)
1  market_place  POINT (377732.671 720748.183)
2          park  POINT (378102.319 719409.507)
3   arts_centre  POINT (377880.409 719423.976)
4         pitch  POINT (377906.133 719672.125)
POINT (376779.99275387503 720951.2728126525)


In [16]:
# Snap each POI to nearest graph node

poi_coords = np.array([
    (geom.x, geom.y) for geom in pois_points.geometry
])

#Query nearest graph node

distances, indices = tree.query (poi_coords, k=1)

# Assign nearest nodes

pois_points["nearest_node"] = [tuple(nodes[i]) for i in indices]

#Confirm

pois_points[["geometry", "nearest_node"]].head()


,geometry,nearest_node
0,POINT (376779.993 720951.273),"(376842.9147142944, 720919.1210646764)"
1,POINT (377732.671 720748.183),"(377726.41298492433, 720729.6651796879)"
2,POINT (378102.319 719409.507),"(378082.3681886849, 719409.2163544127)"
3,POINT (377880.409 719423.976),"(377866.97610594897, 719466.6645263103)"
4,POINT (377906.133 719672.125),"(377946.5806738153, 719658.7936825892)"
